# Notebook 06: Semantic Segmentation Using PyTorch U-Net

**Objective:** Master the theory, architecture, loss formulations, and metrics of pixel-level semantic segmentation for automated visual inspection.

---

### Core Concepts Covered:
1. **Detection vs. Segmentation:** Why bounding boxes overestimate irregular defect areas and why pixel-level masks are essential.
2. **U-Net Architecture & Skip Connections:** How concatenating shallow spatial features with deep semantic representations preserves fine boundary contours.
3. **Extreme Class Imbalance:** Why standard BCE fails when defects occupy only $1\text{--}5\%$ of pixels.
4. **Soft Dice Loss & Combined Loss:** Formulating region-based loss for robust gradient propagation.
5. **Segmentation Metrics:** Formulating Pixel Accuracy, IoU (Jaccard Index), and Dice Coefficient (F1-Score) from first principles.
6. **Inference & Defect Blob Extraction:** Running `UNetSegmenter` to extract connected components and surface area percentages.

In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.segmentation.unet import UNet
from src.evaluation.segmentation_metrics import (
    compute_iou, compute_dice_coefficient, compute_pixel_accuracy,
    SoftDiceLoss, CombinedBCEDiceLoss
)
from src.inference.segmenter_inference import UNetSegmenter

print("Segmentation modules loaded successfully!")

## 1. Why Bounding Boxes Are Insufficient for Precision Inspection

Consider a diagonal hairline scratch on a metal or glass surface:
- A **bounding box** spanning from $(10, 10)$ to $(190, 190)$ encloses an area of $180 \times 180 = 32,400$ pixels ($81\%$ of a $200 \times 200$ image).
- The **actual scratch** is only 2 pixels wide along the diagonal: approximately $180 \times \sqrt{2} \times 2 \approx 509$ pixels ($1.27\%$ of the image).

If automated quality control uses bounding box area to determine defect severity, the product would be rejected as massive scrap when it was only a minor hairline scratch.

**Semantic Segmentation solves this:** Every pixel $(x, y)$ is classified as either defect ($1$) or non-defective background ($0$).

## 2. U-Net Architecture & Skip Connections

As an image passes through consecutive pooling layers in an encoder:
- **Receptive Field increases:** The network learns *what* is in the image (semantic context).
- **Spatial Resolution collapses:** Fine boundary localization is lost ($256 \to 128 \to 64 \to 32 \to 16$).

If we simply upsampled from the bottleneck ($16 \times 16$), the output masks would be blurry blobs with no edge precision.

**The Skip Connection Solution:**
At each decoder level, feature maps from the corresponding encoder stage are **concatenated along the channel dimension**:
$$X_{\text{decoder}} = [X_{\text{upsampled}}, \; X_{\text{encoder}}]$$
This injects high-resolution spatial details directly into the decoder, restoring sharp defect boundaries.

In [ ]:
# Instantiate U-Net and track tensor shapes across encoder and decoder
model = UNet(in_channels=3, num_classes=1, base_channels=32, bilinear=True)
print(f"Total trainable parameters: {model.count_parameters():,}")

dummy_input = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    logits = model(dummy_input)
print(f"Input Shape : {dummy_input.shape}")
print(f"Output Shape: {logits.shape}")

## 3. The Severe Class Imbalance Problem

In industrial inspection, defect pixels typically comprise only $1\% - 5\%$ of total image area.

**Why Binary Cross-Entropy (BCE) Alone Fails:**
If $98\%$ of pixels are background, a trivial model that predicts all zeros achieves $98\%$ accuracy!
The vast sea of background pixels dominates gradient descent, causing the network to ignore small defects.

**Soft Dice Loss:**
$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2 \sum p_i g_i + \epsilon}{\sum p_i^2 + \sum g_i^2 + \epsilon}$$
- Operates on the **region overlap** rather than pixel-by-pixel sums.
- Normalizes by total positive predictions and ground-truth area, treating a 50-pixel defect with the same relative importance as a 5000-pixel defect.

**Combined BCE + Dice Loss:**
$$\mathcal{L} = \mathcal{L}_{\text{BCE}} + \lambda \mathcal{L}_{\text{Dice}}$$
Combines the smooth optimization landscape of BCE with the class-imbalance resilience of Dice Loss.

In [ ]:
# Compare BCE vs Dice Loss under extreme class imbalance (1% defect)
bce_criterion = torch.nn.BCEWithLogitsLoss()
dice_criterion = SoftDiceLoss()

# Create ground truth mask with a tiny defect (100 pixels out of 10,000 = 1%)
target = torch.zeros(1, 1, 100, 100)
target[:, :, 45:55, 45:55] = 1.0

# Trivial baseline: predicting all background (large negative logits)
trivial_logits = torch.full((1, 1, 100, 100), -5.0)
trivial_probs = torch.sigmoid(trivial_logits)

bce_loss_trivial = bce_criterion(trivial_logits, target).item()
dice_loss_trivial = dice_criterion(trivial_probs, target).item()

print(f"Trivial background prediction BCE Loss : {bce_loss_trivial:.4f} (Deceptively small!)")
print(f"Trivial background prediction Dice Loss: {dice_loss_trivial:.4f} (Correctly penalizes failure near 1.0!)")

## 4. First-Principles Segmentation Metrics: IoU vs Dice

- **IoU (Jaccard Index):** $\frac{|A \cap B|}{|A \cup B|}$
- **Dice Coefficient (F1):** $\frac{2 |A \cap B|}{|A| + |B|}$
- Mathematical equivalence: $\text{Dice} = \frac{2 \cdot \text{IoU}}{1 + \text{IoU}}$
- Notice that $\text{Dice} \ge \text{IoU}$ always holds for positive overlap.

In [ ]:
# Mathematical verification of IoU vs Dice relationship
ious = np.linspace(0.0, 1.0, 100)
dices = (2 * ious) / (1 + ious)

plt.figure(figsize=(7, 5))
plt.plot(ious, dices, label="Dice = 2*IoU / (1 + IoU)", color="crimson", linewidth=2)
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="y = x")
plt.xlabel("Intersection over Union (IoU / Jaccard)", fontsize=11)
plt.ylabel("Dice Coefficient (F1-Score)", fontsize=11)
plt.title("Mathematical Relationship: Dice vs IoU", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 5. End-to-End Defect Segmentation & Blob Extraction Demo

In [ ]:
# Initialize production segmenter
segmenter = UNetSegmenter(model_path=None, device="cpu", base_channels=16)

# Simulate inspection image with synthetic defect
test_image = np.ones((256, 256, 3), dtype=np.uint8) * 180
cv2.circle(test_image, (100, 100), 20, (50, 50, 50), -1) # Synthetic defect blob

response, bin_mask, prob_map = segmenter.predict(test_image)
print(f"Inference Latency      : {response.inference_time_ms:.2f} ms")
print(f"Defective Status       : {response.is_defective}")
print(f"Total Defect Pixels    : {response.total_defect_pixels}")
print(f"Defect Surface Ratio   : {response.defect_area_ratio * 100:.3f}%")
print(f"Num Connected Blobs    : {response.num_defect_components}")